# KAMARA — NLP v2
## Avaliação final no conjunto de teste

Esta é a avaliação **única e final** do `test.jsonl` sintético congelado.

O notebook compara:

1. **TF-IDF + Logistic Regression v2**
2. **BERTimbau v2**

Regras metodológicas desta etapa:

- nenhum modelo é treinado;
- nenhum hiperparâmetro é alterado;
- nenhum limiar é ajustado;
- nenhum exemplo é removido ou reclassificado;
- o `test.jsonl` é usado apenas para inferência e relatório;
- os resultados do teste **não devem ser usados para voltar e ajustar os modelos**;
- a avaliação independente/humana/externa continua necessária depois desta etapa.

Arquivos necessários:

```text
test.jsonl
classifier.joblib
vectorizer.joblib
kamara_bertimbau_v2_export.zip
```

Os dois `.joblib` devem ser os artefatos de:

```text
ml/artifacts/nlp/tfidf_logreg_v2/
```

O ZIP deve ser o export já aprovado de:

```text
kamara_bertimbau_v2
```

In [1]:
# 1. Dependências
!pip -q install \
    "transformers>=4.45,<5" \
    "scikit-learn==1.7.2" \
    "pandas>=2,<3" \
    "joblib>=1.4,<2" \
    "safetensors>=0.4,<1" \
    "matplotlib>=3.8,<4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# 2. Imports e ambiente
import hashlib
import json
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
BERT_BATCH_SIZE = 64 if torch.cuda.is_available() else 16

EXPECTED_LABELS = [
    "COBRADE_1_2_1_0_0",
    "COBRADE_1_2_2_0_0",
    "COBRADE_1_2_3_0_0",
    "COBRADE_1_4_1_3_1",
    "COBRADE_1_4_1_3_2",
    "COBRADE_2_3_1_1_0",
    "COBRADE_2_3_1_2_0",
    "OTHER",
    "INSUFFICIENT_INFO",
]

print("Python       :", os.sys.version.split()[0])
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA         :", torch.cuda.is_available())
print("DEVICE       :", DEVICE)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
else:
    print("AVISO: GPU não está ativa. BERTimbau funcionará em CPU, mas será mais lento.")

Python       : 3.13.15
torch        : 2.11.0+cu128
transformers : 4.57.6
scikit-learn: 1.7.2
CUDA         : True
DEVICE       : cuda
GPU          : Tesla T4


## Upload

Envie exatamente os quatro arquivos pedidos.

Este notebook aceita nomes renomeados pelo Colab, como:

```text
test (1).jsonl
classifier (1).joblib
vectorizer (1).joblib
kamara_bertimbau_v2_export (1).zip
```

e os padroniza internamente.

In [3]:
# 3. Upload controlado
from google.colab import files

uploaded = files.upload()
names = list(uploaded.keys())

print("\nArquivos recebidos:")
for name in names:
    print(" -", name)

jsonls = [n for n in names if n.lower().endswith(".jsonl")]
zips = [n for n in names if n.lower().endswith(".zip")]
joblibs = [n for n in names if n.lower().endswith(".joblib")]

if len(jsonls) != 1:
    raise RuntimeError(
        f"Envie exatamente 1 JSONL (test.jsonl). Recebidos: {jsonls}"
    )

if len(zips) != 1:
    raise RuntimeError(
        f"Envie exatamente 1 ZIP (kamara_bertimbau_v2_export.zip). Recebidos: {zips}"
    )

if len(joblibs) != 2:
    raise RuntimeError(
        "Envie exatamente 2 arquivos .joblib: classifier.joblib e vectorizer.joblib. "
        f"Recebidos: {joblibs}"
    )

classifier_candidates = [n for n in joblibs if "classifier" in n.lower()]
vectorizer_candidates = [n for n in joblibs if "vectorizer" in n.lower()]

if len(classifier_candidates) != 1 or len(vectorizer_candidates) != 1:
    raise RuntimeError(
        "Não consegui distinguir classifier.joblib e vectorizer.joblib pelos nomes."
    )

TEST_PATH = Path("/content/test.jsonl")
BERT_ZIP_PATH = Path("/content/kamara_bertimbau_v2_export.zip")
TFIDF_CLASSIFIER_PATH = Path("/content/classifier.joblib")
TFIDF_VECTORIZER_PATH = Path("/content/vectorizer.joblib")

mapping = {
    Path("/content") / jsonls[0]: TEST_PATH,
    Path("/content") / zips[0]: BERT_ZIP_PATH,
    Path("/content") / classifier_candidates[0]: TFIDF_CLASSIFIER_PATH,
    Path("/content") / vectorizer_candidates[0]: TFIDF_VECTORIZER_PATH,
}

for src, dst in mapping.items():
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

print("\nPadronizados:")
for p in [
    TEST_PATH,
    BERT_ZIP_PATH,
    TFIDF_CLASSIFIER_PATH,
    TFIDF_VECTORIZER_PATH,
]:
    print(" -", p)

Saving classifier.joblib to classifier.joblib
Saving kamara_bertimbau_v2_export.zip to kamara_bertimbau_v2_export.zip
Saving test.jsonl to test.jsonl
Saving vectorizer.joblib to vectorizer.joblib

Arquivos recebidos:
 - classifier.joblib
 - kamara_bertimbau_v2_export.zip
 - test.jsonl
 - vectorizer.joblib

Padronizados:
 - /content/test.jsonl
 - /content/kamara_bertimbau_v2_export.zip
 - /content/classifier.joblib
 - /content/vectorizer.joblib


In [4]:
# 4. Funções auxiliares
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_jsonl(path: Path):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                raise RuntimeError(f"Linha vazia em {path}:{line_number}")

            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"JSON inválido em {path}:{line_number}"
                ) from exc

            for field in ("sample_id", "text", "target_label"):
                if field not in row:
                    raise RuntimeError(
                        f"{path}:{line_number} sem campo obrigatório {field!r}"
                    )

            rows.append(row)

    return rows

def compute_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=EXPECTED_LABELS,
            average="macro",
            zero_division=0,
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=EXPECTED_LABELS,
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
    }

def per_class_metrics(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        labels=EXPECTED_LABELS,
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for label in EXPECTED_LABELS:
        d = report[label]
        rows.append({
            "target_label": label,
            "precision": float(d["precision"]),
            "recall": float(d["recall"]),
            "f1_score": float(d["f1-score"]),
            "support": int(d["support"]),
        })

    return pd.DataFrame(rows)

def save_confusion_matrix(y_true, y_pred, csv_path, png_path, title):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=EXPECTED_LABELS,
    )

    cm_df = pd.DataFrame(
        cm,
        index=EXPECTED_LABELS,
        columns=EXPECTED_LABELS,
    )
    cm_df.to_csv(csv_path, encoding="utf-8")

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(cm)

    ax.set_xticks(range(len(EXPECTED_LABELS)))
    ax.set_yticks(range(len(EXPECTED_LABELS)))
    ax.set_xticklabels(EXPECTED_LABELS, rotation=90)
    ax.set_yticklabels(EXPECTED_LABELS)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
            )

    fig.tight_layout()
    fig.savefig(png_path, dpi=180)
    plt.close(fig)

    return cm_df

In [5]:
# 5. Validar o conjunto de teste congelado
test_rows = read_jsonl(TEST_PATH)

if len(test_rows) != 1350:
    raise RuntimeError(
        f"Test possui {len(test_rows)} registros; esperado 1350."
    )

sample_ids = [r["sample_id"] for r in test_rows]

if len(sample_ids) != len(set(sample_ids)):
    raise RuntimeError("Há sample_id duplicado no teste.")

counts = Counter(r["target_label"] for r in test_rows)

if set(counts) != set(EXPECTED_LABELS):
    raise RuntimeError(
        f"Labels inesperados no teste: {dict(counts)}"
    )

if any(counts[label] != 150 for label in EXPECTED_LABELS):
    raise RuntimeError(
        f"Teste não possui 150 registros por classe: {dict(counts)}"
    )

if any(
    "split" in r and r["split"] != "test"
    for r in test_rows
):
    raise RuntimeError(
        "Há registros com campo split diferente de 'test'."
    )

TEST_SHA256 = sha256_file(TEST_PATH)

print("Test records:", len(test_rows))
print("SHA-256     :", TEST_SHA256)
print("Distribuição:")

for label in EXPECTED_LABELS:
    print(f"  {label:<30} {counts[label]}")

print("\nConjunto de teste validado. A partir deste ponto, os resultados são finais.")

Test records: 1350
SHA-256     : 0b7ce417fa9fd9e619b3f04cbfc176d3478342ea5eb2eed1507b50393798477d
Distribuição:
  COBRADE_1_2_1_0_0              150
  COBRADE_1_2_2_0_0              150
  COBRADE_1_2_3_0_0              150
  COBRADE_1_4_1_3_1              150
  COBRADE_1_4_1_3_2              150
  COBRADE_2_3_1_1_0              150
  COBRADE_2_3_1_2_0              150
  OTHER                          150
  INSUFFICIENT_INFO              150

Conjunto de teste validado. A partir deste ponto, os resultados são finais.


In [6]:
# 6. Preparar textos e rótulos verdadeiros
test_texts = [r["text"] for r in test_rows]
true_labels = [r["target_label"] for r in test_rows]

print("Textos:", len(test_texts))
print("Labels:", len(true_labels))

Textos: 1350
Labels: 1350


# TF-IDF + Logistic Regression v2

Nenhum `fit` é executado nesta seção.

In [7]:
# 7. Carregar artefatos TF-IDF v2
tfidf_classifier = joblib.load(TFIDF_CLASSIFIER_PATH)
tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)

if not hasattr(tfidf_classifier, "predict"):
    raise RuntimeError("classifier.joblib não possui método predict.")

if not hasattr(tfidf_vectorizer, "transform"):
    raise RuntimeError("vectorizer.joblib não possui método transform.")

classifier_classes = list(tfidf_classifier.classes_)

if set(classifier_classes) != set(EXPECTED_LABELS):
    raise RuntimeError(
        "Classes do classificador TF-IDF não correspondem ao experimento.\n"
        f"Modelo: {classifier_classes}\n"
        f"Esperado: {EXPECTED_LABELS}"
    )

print("TF-IDF v2 carregado.")
print("Classes:", classifier_classes)

TF-IDF v2 carregado.
Classes: [np.str_('COBRADE_1_2_1_0_0'), np.str_('COBRADE_1_2_2_0_0'), np.str_('COBRADE_1_2_3_0_0'), np.str_('COBRADE_1_4_1_3_1'), np.str_('COBRADE_1_4_1_3_2'), np.str_('COBRADE_2_3_1_1_0'), np.str_('COBRADE_2_3_1_2_0'), np.str_('INSUFFICIENT_INFO'), np.str_('OTHER')]


In [8]:
# 8. Inferência TF-IDF v2
X_test = tfidf_vectorizer.transform(test_texts)

tfidf_pred = tfidf_classifier.predict(X_test)

if hasattr(tfidf_classifier, "predict_proba"):
    tfidf_probs = tfidf_classifier.predict_proba(X_test)
    tfidf_conf = np.max(tfidf_probs, axis=1)
else:
    tfidf_probs = None
    tfidf_conf = np.full(len(test_rows), np.nan)

tfidf_metrics = compute_metrics(
    true_labels,
    tfidf_pred,
)

print(json.dumps(tfidf_metrics, indent=2, ensure_ascii=False))

{
  "accuracy": 0.9955555555555555,
  "macro_precision": 0.9955942784890153,
  "macro_recall": 0.9955555555555556,
  "macro_f1": 0.9955578520364975,
  "weighted_f1": 0.9955578520364974
}


# BERTimbau v2

O modelo é carregado do export já aprovado. Nenhum treinamento é executado.

In [9]:
# 9. Extrair e localizar kamara_bertimbau_v2
EXTRACT_ROOT = Path("/content/kamara_bertimbau_v2_export_extracted")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BERT_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

model_candidates = []

for config_path in EXTRACT_ROOT.rglob("config.json"):
    parent = config_path.parent

    if (
        parent.name == "kamara_bertimbau_v2"
        and (parent / "model.safetensors").is_file()
    ):
        model_candidates.append(parent)

if len(model_candidates) != 1:
    raise RuntimeError(
        "Não foi encontrado exatamente um diretório "
        f"kamara_bertimbau_v2. Encontrados: {model_candidates}"
    )

BERT_MODEL_DIR = model_candidates[0]

print("BERTimbau v2:", BERT_MODEL_DIR)

metadata_path = BERT_MODEL_DIR / "run_metadata.json"

if metadata_path.is_file():
    bert_metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    print("run_name :", bert_metadata.get("run_name"))
    print("test_used:", bert_metadata.get("test_used"))

    if bert_metadata.get("run_name") not in (
        None,
        "kamara_bertimbau_v2",
    ):
        raise RuntimeError(
            "O ZIP não corresponde ao kamara_bertimbau_v2."
        )

    if bert_metadata.get("test_used") is True:
        raise RuntimeError(
            "O metadata informa uso anterior do teste."
        )
else:
    bert_metadata = {}
    print("AVISO: run_metadata.json não encontrado.")

BERTimbau v2: /content/kamara_bertimbau_v2_export_extracted/artifacts/kamara_bertimbau_v2
run_name : kamara_bertimbau_v2
test_used: False


In [10]:
# 10. Carregar BERTimbau v2
tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_DIR
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_DIR
)

bert_model.to(DEVICE)
bert_model.eval()

bert_id2label = {
    int(k): v
    for k, v in bert_model.config.id2label.items()
}

bert_labels = [
    bert_id2label[i]
    for i in sorted(bert_id2label)
]

if bert_labels != EXPECTED_LABELS:
    raise RuntimeError(
        "A ordem das classes do BERTimbau não corresponde ao experimento.\n"
        f"Modelo: {bert_labels}\n"
        f"Esperado: {EXPECTED_LABELS}"
    )

print("BERTimbau v2 carregado.")

BERTimbau v2 carregado.


In [11]:
# 11. Inferência BERTimbau v2
@torch.no_grad()
def bert_predict(texts):
    all_pred_ids = []
    all_conf = []
    all_probs = []

    for start in range(0, len(texts), BERT_BATCH_SIZE):
        batch = texts[start:start + BERT_BATCH_SIZE]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {
            k: v.to(DEVICE)
            for k, v in encoded.items()
        }

        logits = bert_model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)

        conf, pred_ids = torch.max(probs, dim=-1)

        all_pred_ids.extend(pred_ids.cpu().tolist())
        all_conf.extend(conf.cpu().tolist())
        all_probs.extend(probs.cpu().numpy())

    labels = [
        bert_id2label[int(i)]
        for i in all_pred_ids
    ]

    return (
        np.asarray(labels),
        np.asarray(all_conf),
        np.asarray(all_probs),
    )

bert_pred, bert_conf, bert_probs = bert_predict(test_texts)

bert_metrics = compute_metrics(
    true_labels,
    bert_pred,
)

print(json.dumps(bert_metrics, indent=2, ensure_ascii=False))

{
  "accuracy": 0.9940740740740741,
  "macro_precision": 0.9941452616471851,
  "macro_recall": 0.9940740740740741,
  "macro_f1": 0.9940634880016881,
  "weighted_f1": 0.9940634880016879
}


# Resultados finais

In [12]:
# 12. Comparação geral
comparison_df = pd.DataFrame([
    {
        "model": "tfidf_logreg_v2",
        **tfidf_metrics,
        "errors": int(
            np.sum(np.asarray(tfidf_pred) != np.asarray(true_labels))
        ),
    },
    {
        "model": "kamara_bertimbau_v2",
        **bert_metrics,
        "errors": int(
            np.sum(np.asarray(bert_pred) != np.asarray(true_labels))
        ),
    },
])

display(comparison_df)

print("\n=== RESULTADO FINAL — TESTE SINTÉTICO CONGELADO ===")

for _, row in comparison_df.iterrows():
    print(f"\n{row['model']}")
    print(f"  accuracy        : {row['accuracy']:.6f}")
    print(f"  macro_precision : {row['macro_precision']:.6f}")
    print(f"  macro_recall    : {row['macro_recall']:.6f}")
    print(f"  macro_f1        : {row['macro_f1']:.6f}")
    print(f"  weighted_f1     : {row['weighted_f1']:.6f}")
    print(f"  errors          : {int(row['errors'])}")

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,errors
0,tfidf_logreg_v2,0.995556,0.995594,0.995556,0.995558,0.995558,6
1,kamara_bertimbau_v2,0.994074,0.994145,0.994074,0.994063,0.994063,8



=== RESULTADO FINAL — TESTE SINTÉTICO CONGELADO ===

tfidf_logreg_v2
  accuracy        : 0.995556
  macro_precision : 0.995594
  macro_recall    : 0.995556
  macro_f1        : 0.995558
  weighted_f1     : 0.995558
  errors          : 6

kamara_bertimbau_v2
  accuracy        : 0.994074
  macro_precision : 0.994145
  macro_recall    : 0.994074
  macro_f1        : 0.994063
  weighted_f1     : 0.994063
  errors          : 8


In [13]:
# 13. Métricas por classe
tfidf_per_class = per_class_metrics(
    true_labels,
    tfidf_pred,
)

bert_per_class = per_class_metrics(
    true_labels,
    bert_pred,
)

print("TF-IDF v2")
display(tfidf_per_class)

print("\nBERTimbau v2")
display(bert_per_class)

TF-IDF v2


,target_label,precision,recall,f1_score,support
0,COBRADE_1_2_1_0_0,0.986842,1.000000,0.993377,150
1,COBRADE_1_2_2_0_0,0.980263,0.993333,0.986755,150
2,COBRADE_1_2_3_0_0,1.000000,0.993333,0.996656,150
3,COBRADE_1_4_1_3_1,1.000000,1.000000,1.000000,150
4,COBRADE_1_4_1_3_2,1.000000,1.000000,1.000000,150
5,COBRADE_2_3_1_1_0,1.000000,1.000000,1.000000,150
6,COBRADE_2_3_1_2_0,1.000000,1.000000,1.000000,150
7,OTHER,0.993243,0.980000,0.986577,150
8,INSUFFICIENT_INFO,1.000000,0.993333,0.996656,150



BERTimbau v2


,target_label,precision,recall,f1_score,support
0,COBRADE_1_2_1_0_0,0.986842,1.000000,0.993377,150
1,COBRADE_1_2_2_0_0,0.974026,1.000000,0.986842,150
2,COBRADE_1_2_3_0_0,0.993289,0.986667,0.989967,150
3,COBRADE_1_4_1_3_1,1.000000,1.000000,1.000000,150
4,COBRADE_1_4_1_3_2,1.000000,1.000000,1.000000,150
5,COBRADE_2_3_1_1_0,1.000000,1.000000,1.000000,150
6,COBRADE_2_3_1_2_0,1.000000,1.000000,1.000000,150
7,OTHER,0.993151,0.966667,0.979730,150
8,INSUFFICIENT_INFO,1.000000,0.993333,0.996656,150


In [14]:
# 14. Predições e erros
base_columns = {
    "sample_id": [r["sample_id"] for r in test_rows],
    "true_label": true_labels,
    "linguistic_profile": [
        r.get("linguistic_profile")
        for r in test_rows
    ],
    "difficulty": [
        r.get("difficulty")
        for r in test_rows
    ],
    "scenario_family": [
        r.get("scenario_family")
        for r in test_rows
    ],
    "text": test_texts,
}

tfidf_predictions_df = pd.DataFrame({
    **base_columns,
    "predicted_label": tfidf_pred,
    "correct": (
        np.asarray(tfidf_pred) == np.asarray(true_labels)
    ).astype(int),
    "predicted_probability": tfidf_conf,
})

bert_predictions_df = pd.DataFrame({
    **base_columns,
    "predicted_label": bert_pred,
    "correct": (
        np.asarray(bert_pred) == np.asarray(true_labels)
    ).astype(int),
    "predicted_probability": bert_conf,
})

tfidf_errors_df = tfidf_predictions_df[
    tfidf_predictions_df["correct"] == 0
].copy()

bert_errors_df = bert_predictions_df[
    bert_predictions_df["correct"] == 0
].copy()

print("TF-IDF erros:", len(tfidf_errors_df))
display(tfidf_errors_df)

print("\nBERTimbau erros:", len(bert_errors_df))
display(bert_errors_df)

TF-IDF erros: 6


,sample_id,true_label,linguistic_profile,difficulty,scenario_family,text,predicted_label,correct,predicted_probability
180,cobrade_1_2_2_0_0_0214,COBRADE_1_2_2_0_0,formal,typical,rapid_street_runoff,A enxurrada atravessou a área urbana e desloco...,OTHER,0,0.321624
440,cobrade_1_2_3_0_0_0938,COBRADE_1_2_3_0_0,indirect,hard,standing_water_after_rain,"A água está espalhada pela largura da rua, mas...",COBRADE_1_2_2_0_0,0,0.586865
1155,insufficient_info_0716,INSUFFICIENT_INFO,slang,typical,very_short_vague,cara o rio subiu,COBRADE_1_2_1_0_0,0,0.442235
1201,other_v2_0020,OTHER,informal,typical,windstorm,a ventania virou lixeira derrubou galho e deix...,COBRADE_1_2_2_0_0,0,0.354154
1234,other_v2_0220,OTHER,informal,typical,landslide,o talude soltou e veio tudo pra baixo com mato...,COBRADE_1_2_2_0_0,0,0.376654
1267,other_v2_0453,OTHER,neutral,typical,drought,A estiagem reduziu bastante a água disponível ...,COBRADE_1_2_1_0_0,0,0.287636



BERTimbau erros: 8


,sample_id,true_label,linguistic_profile,difficulty,scenario_family,text,predicted_label,correct,predicted_probability
336,cobrade_1_2_3_0_0_0238,COBRADE_1_2_3_0_0,formal,typical,standing_water_after_rain,A lâmina d'água permanece estável sobre o asfa...,OTHER,0,0.596659
440,cobrade_1_2_3_0_0_0938,COBRADE_1_2_3_0_0,indirect,hard,standing_water_after_rain,"A água está espalhada pela largura da rua, mas...",COBRADE_1_2_2_0_0,0,0.974940
1155,insufficient_info_0716,INSUFFICIENT_INFO,slang,typical,very_short_vague,cara o rio subiu,COBRADE_1_2_1_0_0,0,0.955208
1233,other_v2_0210,OTHER,neutral,hard,landslide,"O solo saturado escorregou pelo morro, carrega...",COBRADE_1_2_2_0_0,0,0.656807
1234,other_v2_0220,OTHER,informal,typical,landslide,o talude soltou e veio tudo pra baixo com mato...,COBRADE_1_2_2_0_0,0,0.697877
1235,other_v2_0224,OTHER,informal,hard,landslide,o chão do morro escorregou vários metros e lev...,COBRADE_1_2_2_0_0,0,0.701765
1238,other_v2_0262,OTHER,formal,typical,road_collapse,A rodovia apresenta colapso parcial do leito v...,COBRADE_1_2_3_0_0,0,0.829508
1267,other_v2_0453,OTHER,neutral,typical,drought,A estiagem reduziu bastante a água disponível ...,COBRADE_1_2_1_0_0,0,0.633520


In [15]:
# 15. Comparação pareada dos dois modelos
comparison_predictions_df = pd.DataFrame({
    "sample_id": [r["sample_id"] for r in test_rows],
    "true_label": true_labels,
    "tfidf_prediction": tfidf_pred,
    "bert_prediction": bert_pred,
    "tfidf_correct": (
        np.asarray(tfidf_pred) == np.asarray(true_labels)
    ).astype(int),
    "bert_correct": (
        np.asarray(bert_pred) == np.asarray(true_labels)
    ).astype(int),
    "tfidf_confidence": tfidf_conf,
    "bert_confidence": bert_conf,
    "models_disagree": (
        np.asarray(tfidf_pred) != np.asarray(bert_pred)
    ).astype(int),
    "linguistic_profile": [
        r.get("linguistic_profile")
        for r in test_rows
    ],
    "difficulty": [
        r.get("difficulty")
        for r in test_rows
    ],
    "scenario_family": [
        r.get("scenario_family")
        for r in test_rows
    ],
    "text": test_texts,
})

both_correct = int(
    (
        (comparison_predictions_df["tfidf_correct"] == 1)
        & (comparison_predictions_df["bert_correct"] == 1)
    ).sum()
)

tfidf_only_correct = int(
    (
        (comparison_predictions_df["tfidf_correct"] == 1)
        & (comparison_predictions_df["bert_correct"] == 0)
    ).sum()
)

bert_only_correct = int(
    (
        (comparison_predictions_df["tfidf_correct"] == 0)
        & (comparison_predictions_df["bert_correct"] == 1)
    ).sum()
)

both_wrong = int(
    (
        (comparison_predictions_df["tfidf_correct"] == 0)
        & (comparison_predictions_df["bert_correct"] == 0)
    ).sum()
)

disagreements = int(
    comparison_predictions_df["models_disagree"].sum()
)

paired_summary = {
    "both_correct": both_correct,
    "tfidf_only_correct": tfidf_only_correct,
    "bert_only_correct": bert_only_correct,
    "both_wrong": both_wrong,
    "prediction_disagreements": disagreements,
}

print(json.dumps(paired_summary, indent=2))

display(
    comparison_predictions_df[
        comparison_predictions_df["models_disagree"] == 1
    ]
)

{
  "both_correct": 1340,
  "tfidf_only_correct": 4,
  "bert_only_correct": 2,
  "both_wrong": 4,
  "prediction_disagreements": 6
}


,sample_id,true_label,tfidf_prediction,bert_prediction,tfidf_correct,bert_correct,tfidf_confidence,bert_confidence,models_disagree,linguistic_profile,difficulty,scenario_family,text
180,cobrade_1_2_2_0_0_0214,COBRADE_1_2_2_0_0,OTHER,COBRADE_1_2_2_0_0,0,1,0.321624,0.987019,1,formal,typical,rapid_street_runoff,A enxurrada atravessou a área urbana e desloco...
336,cobrade_1_2_3_0_0_0238,COBRADE_1_2_3_0_0,COBRADE_1_2_3_0_0,OTHER,1,0,0.552453,0.596659,1,formal,typical,standing_water_after_rain,A lâmina d'água permanece estável sobre o asfa...
1201,other_v2_0020,OTHER,COBRADE_1_2_2_0_0,OTHER,0,1,0.354154,0.990527,1,informal,typical,windstorm,a ventania virou lixeira derrubou galho e deix...
1233,other_v2_0210,OTHER,OTHER,COBRADE_1_2_2_0_0,1,0,0.556949,0.656807,1,neutral,hard,landslide,"O solo saturado escorregou pelo morro, carrega..."
1235,other_v2_0224,OTHER,OTHER,COBRADE_1_2_2_0_0,1,0,0.622571,0.701765,1,informal,hard,landslide,o chão do morro escorregou vários metros e lev...
1238,other_v2_0262,OTHER,OTHER,COBRADE_1_2_3_0_0,1,0,0.427309,0.829508,1,formal,typical,road_collapse,A rodovia apresenta colapso parcial do leito v...


In [16]:
# 16. Criar diretórios de saída
EXPORT_ROOT = Path("/content/kamara_nlp_v2_final_test")
TFIDF_DIR = (
    EXPORT_ROOT
    / "evaluation"
    / "tfidf_logreg_v2"
)
BERT_DIR = (
    EXPORT_ROOT
    / "evaluation"
    / "kamara_bertimbau_v2"
)
COMPARISON_DIR = (
    EXPORT_ROOT
    / "evaluation"
    / "comparison_v2"
)

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)

for p in [TFIDF_DIR, BERT_DIR, COMPARISON_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Saída:", EXPORT_ROOT)

Saída: /content/kamara_nlp_v2_final_test


In [17]:
# 17. Salvar resultados TF-IDF
tfidf_predictions_df.to_csv(
    TFIDF_DIR / "predictions_test.csv",
    index=False,
    encoding="utf-8",
)

tfidf_errors_df.to_csv(
    TFIDF_DIR / "errors_test.csv",
    index=False,
    encoding="utf-8",
)

tfidf_per_class.to_csv(
    TFIDF_DIR / "per_class_metrics_test.csv",
    index=False,
    encoding="utf-8",
)

save_confusion_matrix(
    true_labels,
    tfidf_pred,
    TFIDF_DIR / "confusion_matrix_test.csv",
    TFIDF_DIR / "confusion_matrix_test.png",
    "KAMARA TF-IDF + Logistic Regression v2 — Test",
)

tfidf_report = {
    "split": "test",
    "records": len(test_rows),
    "test_sha256": TEST_SHA256,
    "model": "tfidf_logreg_v2",
    **tfidf_metrics,
    "errors": len(tfidf_errors_df),
    "labels": EXPECTED_LABELS,
    "final_test_evaluation": True,
}

with (
    TFIDF_DIR / "metrics_test.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        tfidf_report,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

In [18]:
# 18. Salvar resultados BERTimbau
bert_predictions_df.to_csv(
    BERT_DIR / "predictions_test.csv",
    index=False,
    encoding="utf-8",
)

bert_errors_df.to_csv(
    BERT_DIR / "errors_test.csv",
    index=False,
    encoding="utf-8",
)

bert_per_class.to_csv(
    BERT_DIR / "per_class_metrics_test.csv",
    index=False,
    encoding="utf-8",
)

save_confusion_matrix(
    true_labels,
    bert_pred,
    BERT_DIR / "confusion_matrix_test.csv",
    BERT_DIR / "confusion_matrix_test.png",
    "KAMARA BERTimbau v2 — Test",
)

bert_report = {
    "split": "test",
    "records": len(test_rows),
    "test_sha256": TEST_SHA256,
    "model": "kamara_bertimbau_v2",
    **bert_metrics,
    "errors": len(bert_errors_df),
    "labels": EXPECTED_LABELS,
    "max_length": MAX_LENGTH,
    "final_test_evaluation": True,
}

with (
    BERT_DIR / "metrics_test.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        bert_report,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

In [19]:
# 19. Salvar comparação final
comparison_df.to_csv(
    COMPARISON_DIR / "model_comparison_test.csv",
    index=False,
    encoding="utf-8",
)

comparison_predictions_df.to_csv(
    COMPARISON_DIR / "paired_predictions_test.csv",
    index=False,
    encoding="utf-8",
)

final_summary = {
    "experiment": "kamara_nlp_v2_final_synthetic_test",
    "test_sha256": TEST_SHA256,
    "test_records": len(test_rows),
    "class_support": dict(counts),
    "models": {
        "tfidf_logreg_v2": tfidf_report,
        "kamara_bertimbau_v2": bert_report,
    },
    "paired_comparison": paired_summary,
    "methodological_note": (
        "Avaliação final no conjunto de teste sintético congelado. "
        "Nenhum treinamento, ajuste de hiperparâmetro, ajuste de limiar, "
        "curadoria ou seleção de checkpoint foi realizado com base no teste. "
        "Os resultados não substituem avaliação independente em relatos "
        "humanos ou dados externos."
    ),
}

with (
    COMPARISON_DIR / "final_test_summary.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        final_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

print(json.dumps(final_summary, indent=2, ensure_ascii=False))

{
  "experiment": "kamara_nlp_v2_final_synthetic_test",
  "test_sha256": "0b7ce417fa9fd9e619b3f04cbfc176d3478342ea5eb2eed1507b50393798477d",
  "test_records": 1350,
  "class_support": {
    "COBRADE_1_2_1_0_0": 150,
    "COBRADE_1_2_2_0_0": 150,
    "COBRADE_1_2_3_0_0": 150,
    "COBRADE_1_4_1_3_1": 150,
    "COBRADE_1_4_1_3_2": 150,
    "COBRADE_2_3_1_1_0": 150,
    "COBRADE_2_3_1_2_0": 150,
    "INSUFFICIENT_INFO": 150,
    "OTHER": 150
  },
  "models": {
    "tfidf_logreg_v2": {
      "split": "test",
      "records": 1350,
      "test_sha256": "0b7ce417fa9fd9e619b3f04cbfc176d3478342ea5eb2eed1507b50393798477d",
      "model": "tfidf_logreg_v2",
      "accuracy": 0.9955555555555555,
      "macro_precision": 0.9955942784890153,
      "macro_recall": 0.9955555555555556,
      "macro_f1": 0.9955578520364975,
      "weighted_f1": 0.9955578520364974,
      "errors": 6,
      "labels": [
        "COBRADE_1_2_1_0_0",
        "COBRADE_1_2_2_0_0",
        "COBRADE_1_2_3_0_0",
        "COBRADE

In [20]:
# 20. Listar arquivos produzidos
for path in sorted(EXPORT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(EXPORT_ROOT))

evaluation/comparison_v2/final_test_summary.json
evaluation/comparison_v2/model_comparison_test.csv
evaluation/comparison_v2/paired_predictions_test.csv
evaluation/kamara_bertimbau_v2/confusion_matrix_test.csv
evaluation/kamara_bertimbau_v2/confusion_matrix_test.png
evaluation/kamara_bertimbau_v2/errors_test.csv
evaluation/kamara_bertimbau_v2/metrics_test.json
evaluation/kamara_bertimbau_v2/per_class_metrics_test.csv
evaluation/kamara_bertimbau_v2/predictions_test.csv
evaluation/tfidf_logreg_v2/confusion_matrix_test.csv
evaluation/tfidf_logreg_v2/confusion_matrix_test.png
evaluation/tfidf_logreg_v2/errors_test.csv
evaluation/tfidf_logreg_v2/metrics_test.json
evaluation/tfidf_logreg_v2/per_class_metrics_test.csv
evaluation/tfidf_logreg_v2/predictions_test.csv


In [21]:
# 21. Criar ZIP final
zip_path = shutil.make_archive(
    "/content/kamara_nlp_v2_final_test",
    "zip",
    root_dir=EXPORT_ROOT,
)

print("ZIP criado:", zip_path)
print(
    "Tamanho (KB):",
    round(Path(zip_path).stat().st_size / 1024, 2),
)

ZIP criado: /content/kamara_nlp_v2_final_test.zip
Tamanho (KB): 342.89


In [22]:
# 22. Download
from google.colab import files

files.download(
    "/content/kamara_nlp_v2_final_test.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Depois desta execução

Este teste passa a ser considerado **consumido** para a rodada v2.

Não use seus resultados para:

- alterar exemplos do dataset;
- regenerar uma classe;
- mudar hiperparâmetros;
- trocar checkpoint;
- escolher um novo threshold;
- retreinar especificamente para melhorar o teste.

A próxima etapa científica deve usar um conjunto **independente** do processo
de geração sintética, idealmente relatos humanos revisados ou dados externos.